# step A-2 — scaleup-500 (RQ1 외적 타당성, 실코드, 500 확대 재실행)

**대응 RQ:** RQ1 외적 타당성 — 합성이 아니라 **실제 파일**을 선행 코드로 써서, 언어 관습을 자연 실험으로 이용.

**언어 관습 2×2** (지침만 배정, 실코드는 무조작):

| 선행 파일 | 지침=camel | 지침=snake |
|---|---|---|
| Python (자연 snake) | 충돌 | 일치 |
| JavaScript (자연 camel) | 일치 | 충돌 |

충돌 칸에서 준수율이 낮으면 → 모델이 지침보다 **언어 관습(형태)** 을 따름. 출력이 **행(언어)** 으로 갈리는가 **열(지침)** 으로 갈리는가로 판별.

**파일럿 대비:** 실파일 6개 → **77개(python 37 + javascript 40)로 확장**. seed는 변량이 안 되므로(지침 바꿔도 출력 동일) **파일 수로 500**을 채운다(77 × 2지침 × 3생성 = 462관측 ≈ 500).

**코드 동일 보장(재현성):** 파일럿과 **똑같은 `harness.run`을 호출**만 한다. 기존 6파일 조건은 파일럿(`results/stepA-2/`)과 동일.

설계 문서: `docs/stepA-2/scaleup-500.md`. 출처·라이선스: `data/repo_files/SOURCE.md`.

> **데이터:** 이 노트북은 `data/repo_files/{python,javascript}/`를 **자동 glob**한다. 실파일 번들 완료(77파일 → 154조건). 출처·라이선스: `SOURCE.md`.
> 3B fp16, 생성률만 측정, 재개 가능.

In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepA-2/scaleup-500
!git checkout stepA-2/scaleup-500
!git pull --quiet origin stepA-2/scaleup-500
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — 언어 × 지침 × 실파일(자동 glob). 파일을 번들한 만큼 자동 확장.
import glob
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Source, InstructionForm, Notation)

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
STEP = 'stepA-2_scaleup500'

# data/repo_files/ 기준 상대경로로 파일을 자동 수집(번들한 만큼 늘어난다)
def rel(paths): return sorted(p.split('data/repo_files/', 1)[1] for p in paths)
FILES = {
    'python':     rel(glob.glob('data/repo_files/python/*.py')),
    'javascript': rel(glob.glob('data/repo_files/javascript/*.js')),
}
INSTR = [Notation.CAMEL, Notation.SNAKE]

def make(lang, f, target):
    return Condition(
        model=MODEL,
        preceding=PrecedingCode(n_compliant=0, source=Source.REPO, repo_lang=lang, repo_file=f),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target),
        seed=0,
    )

conditions = [make(lang, f, t) for lang, fs in FILES.items() for f in fs for t in INSTR]

PREDICTION = ('일치 칸(파일 관습=지침) 높음, 충돌 칸 낮음이면 언어 관습(형태)이 지침을 압도 → RQ1 외적 타당성. '
              '출력이 행(언어)으로 갈림 예상. 기존 6파일 조건은 파일럿과 동일.')

n_py, n_js = len(FILES['python']), len(FILES['javascript'])
print(f'파일: python {n_py} + javascript {n_js} = {n_py + n_js}개')
print(f'조건: {len(conditions)} = ({n_py}+{n_js}) x 2 지침  →  생성함수 ≈ {len(conditions) * 3}관측')
if n_py + n_js <= 6:
    print('※ 아직 파일럿 규모(6파일). 500을 채우려면 data/repo_files/ 에 실파일을 더 번들하라.')

In [ ]:
# 실행 — 조건별 순차 생성 + 즉시 저장(재개) + 중간 결과 실시간 표시. 원문 자동 저장.
import pandas as pd
from IPython.display import clear_output
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model

handle = load_model(MODEL)
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info())

NAT = {'python': 'snake', 'javascript': 'camel'}   # 언어 자연 관습

def _rows(cond, metrics):
    lang = cond.preceding.repo_lang; instr = cond.instruction.target_notation.value
    return [{'lang': lang, 'instr': instr, 'compliant': nt == instr}
            for nt in metrics.extra['turn_notations']]

def _show(live, done, total, new, skipped):
    df = pd.DataFrame(live)
    piv = df.groupby(['lang', 'instr'])['compliant'].mean().unstack().round(3)
    clear_output(wait=True)
    print(f'진행 {done}/{total}  (새로 {new} / 건너뜀 {skipped})')
    print('준수율 2x2 (행=파일 언어, 열=지침) — 지금까지:')
    print(piv)
    print('자연 관습: python=snake, javascript=camel → 대각선 반대편이 충돌 칸')

live, new, skipped = [], 0, 0
for i, c in enumerate(conditions):
    p = result_path(c, step=STEP)
    if p.exists():                                   # 이미 한 조건 → 건너뜀(재개)
        live += _rows(c, load_result(p).metrics); skipped += 1
    else:
        out = run(c, handle=handle)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ1', prediction=PREDICTION))
        live += _rows(out.condition, out.metrics); new += 1
    if (i + 1) % 8 == 0 or (i + 1) == len(conditions):
        _show(live, i + 1, len(conditions), new, skipped)
print(f'완료: 새로 {new}, 건너뜀 {skipped}, 총 {len(conditions)}')

In [ ]:
# 결과 로드 — results/stepA-2_scaleup500/
from harness import result_path
from harness.results import load_result

records = [load_result(result_path(c, step=STEP)) for c in conditions]
print('로드:', len(records), '건 → results/'+STEP+'/')

In [ ]:
# 요약 — 2x2 준수율 (언어 x 지침). 생성된 모든 함수(턴) 기준.
import pandas as pd, matplotlib.pyplot as plt

rows = []
for r in records:
    c = r.condition; tgt = c.instruction.target_notation.value
    for nt in r.metrics.extra['turn_notations']:
        rows.append({'lang': c.preceding.repo_lang, 'instr': tgt,
                     'compliant': nt == tgt, 'notation': nt})
df = pd.DataFrame(rows)

piv = df.groupby(['lang', 'instr'])['compliant'].mean().unstack().round(3)
print('준수율 2x2 (행=파일 언어, 열=지침 목표표기):'); print(piv)
print('\n자연 관습: python=snake, javascript=camel → 대각선 반대편이 충돌 칸')

# 출력이 행(언어)으로 갈리는지 열(지침)으로 갈리는지: 언어별 실제 표기 분포
print('\n언어별 생성 표기 분포(지침 무관):')
print(df.groupby('lang')['notation'].value_counts(normalize=True).round(3))

ax = piv.plot(kind='bar', figsize=(5,3))
ax.set_ylabel('준수율(생성)'); ax.set_ylim(0,1); ax.set_title('RQ1 외적타당성: 언어 x 지침 (scaleup-500)')
ax.legend(title='지침'); plt.grid(axis='y', alpha=.3); plt.tight_layout(); plt.show()